# 7장 규칙 기반 판정과 LED 신호

이 노트북은 웹 교재 [7장 규칙 기반 판정과 LED 신호](https://pidocs.aifusionlab.org/ai-python/ch07/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/seoul_weather_2021_2025.csv
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/class_rssi_20261104.csv
!ls *.csv

## 7.1 임계값 설계

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch07/7-1/)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("seoul_weather_2021_2025.csv", parse_dates=["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
season_of = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
             6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
df["season"] = df["month"].map(season_of)
order = ["DJF", "MAM", "JJA", "SON"]
rssi = pd.read_csv("class_rssi_20261104.csv")

In [ ]:
th = df.groupby("season")["temp_mean"].quantile([0.1, 0.9]).unstack().loc[order]
th.round(1)

In [ ]:
def judge_temp(season, t):
    """계절별 분위수 임계값으로 일평균기온을 판정한다."""
    if t < th.loc[season, 0.1]:
        return "추움"
    if t > th.loc[season, 0.9]:
        return "더움"
    return "보통"

print(judge_temp("JJA", 17.0), judge_temp("SON", 17.0))
df["level"] = [judge_temp(s, t) for s, t in zip(df["season"], df["temp_mean"])]
pd.crosstab(df["season"], df["level"]).loc[order, ["추움", "보통", "더움"]]

In [ ]:
q = rssi.groupby("zone")["rssi"].quantile([0.1, 0.25]).unstack()
print(q)

def judge_rssi(zone, v):
    """지점별 분위수 임계값으로 신호 세기를 3단계 판정한다."""
    if v < q.loc[zone, 0.1]:
        return "경고"
    if v < q.loc[zone, 0.25]:
        return "주의"
    return "정상"

rssi["level"] = [judge_rssi(z, v) for z, v in zip(rssi["zone"], rssi["rssi"])]
pd.crosstab(rssi["zone"], rssi["level"])[["정상", "주의", "경고"]]

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=True)
for ax, z in zip(axes, ["D1", "D2", "D3", "D4"]):
    ax.hist(rssi[rssi["zone"] == z]["rssi"], bins=range(-90, -30, 3), color="lightgray")
    ax.axvline(q.loc[z, 0.25], color="orange", label="25%")
    ax.axvline(q.loc[z, 0.1], color="red", label="10%")
    ax.set_title(z)
    ax.set_xlabel("RSSI (dBm)")
axes[0].set_ylabel("records")
axes[0].legend()
plt.tight_layout()
plt.show()

## 7.2 z-점수 이상 탐지

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch07/7-2/)

In [ ]:
print(rssi.groupby("zone")["rssi"].agg(["mean", "std"]).round(1))
rssi["z"] = rssi.groupby("zone")["rssi"].transform(lambda s: (s - s.mean()) / s.std())
out = rssi[rssi["z"].abs() >= 3]
out[["node", "zone", "rssi", "z"]].round({"z": 2})

In [ ]:
w = 14
df["roll_mean"] = df["temp_mean"].rolling(w).mean().shift(1)
df["roll_std"] = df["temp_mean"].rolling(w).std().shift(1)
df["z"] = (df["temp_mean"] - df["roll_mean"]) / df["roll_std"]

y25 = df[df["year"] == 2025]
alert = y25[y25["z"].abs() >= 3]
print(len(alert), "일")
alert[["date", "temp_mean", "roll_mean", "roll_std", "z"]].round({"roll_mean": 2, "roll_std": 2, "z": 2})

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.plot(y25["date"], y25["temp_mean"], lw=1, label="daily mean")
plt.plot(y25["date"], y25["roll_mean"], lw=1, label="14-day mean")
plt.fill_between(y25["date"], y25["roll_mean"] - 3 * y25["roll_std"],
                 y25["roll_mean"] + 3 * y25["roll_std"], alpha=0.2, label="mean +/- 3 std")
plt.scatter(alert["date"], alert["temp_mean"], color="red", zorder=3, label="|z| >= 3")
plt.ylabel("temperature (C)")
plt.legend(ncol=4, fontsize=8)
plt.show()

## 7.3 오경보와 미검출

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch07/7-3/)

In [ ]:
df["rain"] = df["precip_mm"] >= 1.0
print(df["rain"].sum(), "일")
df.groupby("rain")["humidity"].describe().round(1)

In [ ]:
alarm = df["humidity"] >= 80
pd.crosstab(df["rain"], alarm, rownames=["rain"], colnames=["alarm"])

In [ ]:
rows = []
for t in [70, 75, 80, 85, 90]:
    alarm = df["humidity"] >= t
    false_alarm = (alarm & ~df["rain"]).sum()   # 오경보
    miss = (~alarm & df["rain"]).sum()          # 미검출
    rows.append({"threshold": t, "false_alarm": false_alarm, "miss": miss})
res = pd.DataFrame(rows)
res

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(res["threshold"], res["false_alarm"], marker="o", label="false alarm")
plt.plot(res["threshold"], res["miss"], marker="o", label="miss")
plt.xlabel("humidity threshold (%)")
plt.ylabel("days")
plt.legend()
plt.show()

## 7.4 실습: 소음 단계 LED 스트립 신호등

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch07/7-4/)

In [ ]:
N = 12          # LED 개수
CAUTION = 40    # 주의 임계값
WARN = 70       # 경고 임계값

def judge_loud(loud):
    """소리 크기(0~100)를 단계 이름과 LED 색 약어로 변환한다."""
    if loud >= WARN:
        return "warn", "r"
    if loud >= CAUTION:
        return "caution", "y"
    return "ok", "g"

def strip_colors(loud):
    """점등 개수는 소리 크기에 비례, 색은 단계로 결정한 led_show 문자열을 반환한다."""
    stage, c = judge_loud(loud)
    n = max(1, round(loud / 100 * N))
    return stage, " ".join([c] * n + ["k"] * (N - n))

for v in [5, 39, 40, 69, 70, 100]:
    print(v, *strip_colors(v))

In [ ]:
samples = [35, 88, 36, 72, 75, 80, 83, 41, 30]   # 예시: 순간 소음 1회(88), 지속 소음 4회(72~83)
count = 0
for v in samples:
    count = count + 1 if v >= WARN else 0
    stage = judge_loud(v)[0]
    if stage == "warn" and count < 3:
        stage = "caution"                          # 연속 3회 미만이면 주의로 판정
    print(v, count, stage)